In [1]:
import pybamm
import matplotlib.pyplot as plt

In [2]:
model = pybamm.lithium_ion.SPM({"SEI": "ec reaction limited"})
parameter_values = pybamm.ParameterValues("Mohtat2020")

def cycle(c_rate):
    return [
        (
            pybamm.step.c_rate(c_rate, termination="2.5V"),
            pybamm.step.c_rate(0, duration=60*30),
            pybamm.step.c_rate(-1, termination="4.2V"),
            pybamm.step.voltage(4.2, termination="0.05C"),
            pybamm.step.c_rate(0, duration=60*30),
        )
    ]

experiment = []
sols = {}
pybamm.set_logging_level("NOTICE")
for c_rate in [1/2, 1, 2]:
    experiment += cycle(c_rate) * 1000

    experiment = pybamm.Experiment(experiment)
    solver = pybamm.IDAKLUSolver()
    sim = pybamm.Simulation(model, parameter_values=parameter_values, experiment=experiment, solver=solver)
    sol = sim.solve()
    sols[c_rate] = sol


2024-09-25 12:51:27.396 - [NOTICE] logger.func(7): Cycle 1/1000 (32.834 us elapsed) --------------------
2024-09-25 12:51:27.396 - [NOTICE] logger.func(7): Cycle 1/1000, step 1/5: Step(0.5, duration=14400.0, termination=[<pybamm.experiment.step.step_termination.VoltageTermination object at 0x1744cfa90>], direction=Discharge)


AttributeError: module 'pybamm.solvers.idaklu' has no attribute 'create_casadi_solver_group'

In [ ]:
pybamm.dynamic_plot(
    list(sols.values()),
    ["Voltage [V]", "Negative electrode SEI thickness [m]"],
    labels=sols.keys(),
)